In [1]:
import pandas as pd

df = pd.read_csv("../docs/resort_attributes.csv")
df

,resort,lat,lon,piste_km,lifts,base_alt,top_alt,vertical_drop,km_blue,km_red,km_black,snowmaking_share,family_friendly,freeride,parks,source_url,collected_on
0,Engelberg-Titlis,46.8162,8.3957,70,17,1003,3020,2017,18,42.5,9.5,0.4,1,1,0,https://www.skiresort.info/ski-resort/titlis-e...,2026-11-11


In [2]:
df.dtypes

resort               object
lat                 float64
lon                 float64
piste_km              int64
lifts                 int64
base_alt              int64
top_alt               int64
vertical_drop         int64
km_blue               int64
km_red              float64
km_black            float64
snowmaking_share    float64
family_friendly       int64
freeride              int64
parks                 int64
source_url           object
collected_on         object
dtype: object

In [4]:
# vertical drop = top - base?
print(df["vertical_drop"] == df["top_alt"] - df["base_alt"])

# blue + red + black km = total piste km?
print(df[["km_blue", "km_red", "km_black"]].sum(axis=1), df["piste_km"])

0    True
dtype: bool
0    70.0
dtype: float64 0    70
Name: piste_km, dtype: int64


## 8.2 Prices (Pricenow API)

### Step 3: test the request in the notebook

In [1]:
import json
import requests

URL = "https://api.pricenow.dev/api/booking/products/180000/journey/validities"

HEADERS = {
    "pratiq-channel-uuid": "dbe49cb5-8c06-498a-901c-fbf65d1166e5",
    "pratiq-api-version": "2025-02-18",
    "pratiq-superjson": "true",
    "accept-language": "en",
    "content-type": "application/json",
    "Origin": "https://shop.titlis.ch",
    "Referer": "https://shop.titlis.ch/",
    "User-Agent": "SkiGuru student project (UZH, non-commercial)",
}

form_state = {
    "allocation": {"kind": "persontype",
                   "values": [{"kind": "anonymous", "quantity": 1,
                               "personType": "adult_age20to999"}]},
    "attributes": {"duration": "1d"},
}

params = {
    "from": "2026-12-01T00:00:00.000+01:00",
    "to": "2026-12-31T23:59:59.999+01:00",
    "formState": json.dumps(form_state, separators=(",", ":")),
}

r = requests.get(URL, headers=HEADERS, params=params, timeout=20)
print(r.status_code)
print(r.text[:2000])

200
{"json":{"data":{"options":[{"availableCapacity":{"kind":"unlimited"},"value":{"from":"2026-11-30T23:00:00.000Z","to":"2026-12-01T22:59:59.999Z"},"label":"1 Tag und 1/2 Tag ab 12:00 Uhr","isAvailable":true,"price":{"minPrice":5400,"maxPrice":5400}},{"availableCapacity":{"kind":"unlimited"},"value":{"from":"2026-11-30T23:00:00.000Z","to":"2026-12-02T22:59:59.999Z"},"label":"2 Tage","isAvailable":true,"price":{"minPrice":10750,"maxPrice":10750}},{"availableCapacity":{"kind":"unlimited"},"value":{"from":"2026-11-30T23:00:00.000Z","to":"2026-12-03T22:59:59.999Z"},"label":"3 Tage","isAvailable":true,"price":{"minPrice":16100,"maxPrice":16100}},{"availableCapacity":{"kind":"unlimited"},"value":{"from":"2026-11-30T23:00:00.000Z","to":"2026-12-04T22:59:59.999Z"},"label":"4 Tage","isAvailable":true,"price":{"minPrice":20400,"maxPrice":20400}},{"availableCapacity":{"kind":"unlimited"},"value":{"from":"2026-11-30T23:00:00.000Z","to":"2026-12-05T22:59:59.999Z"},"label":"5 Tage","isAvailable":t

### Step 4: turn the response into a table and keep only day passes

In [2]:
import pandas as pd

options = r.json()["json"]["data"]["options"]
raw = pd.json_normalize(options)

# UTC → Swiss time
raw["start"] = pd.to_datetime(raw["value.from"]).dt.tz_convert("Europe/Zurich")
raw["end"] = pd.to_datetime(raw["value.to"]).dt.tz_convert("Europe/Zurich")

# number of days the ticket is valid
raw["n_days"] = (raw["end"].dt.normalize() - raw["start"].dt.normalize()).dt.days + 1

day = raw[raw["n_days"] == 1].copy()
day["date"] = day["start"].dt.date
day["price"] = day["price.minPrice"] / 100

print(raw.shape)
print(day["label"].unique())
day[["date", "price", "price.maxPrice", "isAvailable"]]

(434, 10)
['1 Tag und 1/2 Tag ab 12:00 Uhr']


,date,price,price.maxPrice,isAvailable
0,2026-12-01,54.0,5400,True
14,2026-12-02,53.5,5350,True
28,2026-12-03,53.5,5350,True
42,2026-12-04,53.5,5350,True
56,2026-12-05,61.0,6100,True
70,2026-12-06,61.5,6150,True
84,2026-12-07,53.5,5350,True
98,2026-12-08,56.0,5600,True
112,2026-12-09,53.0,5300,True
126,2026-12-10,53.5,5350,True


### Step 5: fetch all months of the season (Dec-Apr) in a loop

In [5]:
import time
from datetime import date

def fetch_month(year, month):
    """Day-pass prices (CHF) for one month at Engelberg-Titlis."""
    start = pd.Timestamp(year=year, month=month, day=1, tz="Europe/Zurich")
    end = start + pd.DateOffset(months=1) - pd.Timedelta(milliseconds=1)
    params = {
        "from": start.to_pydatetime().isoformat(timespec="milliseconds"),
        "to": end.to_pydatetime().isoformat(timespec="milliseconds"),
        "formState": json.dumps(form_state, separators=(",", ":")),
    }
    r = requests.get(URL, headers=HEADERS, params=params, timeout=20)
    r.raise_for_status()
    options = r.json()["json"]["data"]["options"]
    if not options:                      # e.g. after the end of the season
        return pd.DataFrame(columns=["date", "price"])

    raw = pd.json_normalize(options)
    raw["start"] = pd.to_datetime(raw["value.from"]).dt.tz_convert("Europe/Zurich")
    raw["end"] = pd.to_datetime(raw["value.to"]).dt.tz_convert("Europe/Zurich")
    raw["n_days"] = (raw["end"].dt.normalize() - raw["start"].dt.normalize()).dt.days + 1

    day = raw[raw["label"].str.startswith("1 Tag ") & raw["isAvailable"]].copy()
    day["date"] = day["start"].dt.date
    day["price"] = day["price.minPrice"] / 100
    return day[["date", "price"]]


months = [(2026, 12), (2027, 1), (2027, 2), (2027, 3), (2027, 4)]
frames = []
for y, m in months:
    frames.append(fetch_month(y, m))
    time.sleep(2)                        # pause between requests (politeness)

prices = pd.concat(frames, ignore_index=True)
prices["resort"] = "Engelberg-Titlis"
prices["scraped_on"] = date.today()
prices["days_ahead"] = (pd.to_datetime(prices["date"]) - pd.to_datetime(prices["scraped_on"])).dt.days
prices = prices[["resort", "date", "price", "scraped_on", "days_ahead"]]

print(prices.shape)
print(prices.groupby(pd.to_datetime(prices["date"]).dt.month)["price"].agg(["count", "min", "max"]))
prices.head()

(151, 5)
      count   min   max
date                   
1        31  54.5  67.5
2        28  62.5  68.5
3        31  54.0  65.5
4        30  51.0  59.0
12       31  53.0  69.0


,resort,date,price,scraped_on,days_ahead
0,Engelberg-Titlis,2026-12-01,54.0,2026-11-16,15
1,Engelberg-Titlis,2026-12-02,53.5,2026-11-16,16
2,Engelberg-Titlis,2026-12-03,53.5,2026-11-16,17
3,Engelberg-Titlis,2026-12-04,53.5,2026-11-16,18
4,Engelberg-Titlis,2026-12-05,61.0,2026-11-16,19


#### Step 5b: find duplicate and outlier

In [4]:
# 1. dates that appear more than once
print(prices[prices["date"].duplicated(keep=False)])

# 2. the five highest prices
print(prices.sort_values("price", ascending=False).head(5))

# 3. number of unique dates
print(prices["date"].nunique(), len(prices))

               resort        date  price  scraped_on  days_ahead
117  Engelberg-Titlis  2027-03-28   61.5  2026-11-16         132
118  Engelberg-Titlis  2027-03-28  120.5  2026-11-16         132
               resort        date  price  scraped_on  days_ahead
118  Engelberg-Titlis  2027-03-28  120.5  2026-11-16         132
28   Engelberg-Titlis  2026-12-29   69.0  2026-11-16          43
29   Engelberg-Titlis  2026-12-30   69.0  2026-11-16          44
30   Engelberg-Titlis  2026-12-31   69.0  2026-11-16          45
75   Engelberg-Titlis  2027-02-14   68.5  2026-11-16          90
151 152


## 8.3 Train (transport.opendata.ch)

In [6]:
url = "https://transport.opendata.ch/v1/connections"
params = {"from": "Zürich HB", "to": "Engelberg", "date": "2026-12-10", "time": "07:00", "limit": 4}

r = requests.get(url, params=params, timeout=20)
print(r.status_code)

for c in r.json()["connections"]:
    print(c["from"]["departure"], "→", c["to"]["arrival"], "| duration:", c["duration"], "| changes:", c["transfers"])

200
2026-12-10T07:10:00+0100 → 2026-12-10T08:53:00+0100 | duration: 00d01:43:00 | changes: 1
2026-12-10T08:10:00+0100 → 2026-12-10T09:53:00+0100 | duration: 00d01:43:00 | changes: 1
2026-12-10T09:10:00+0100 → 2026-12-10T10:53:00+0100 | duration: 00d01:43:00 | changes: 1
2026-12-10T10:10:00+0100 → 2026-12-10T11:53:00+0100 | duration: 00d01:43:00 | changes: 1


In [7]:
def parse_duration(s):
    """'00d01:43:00' → 103 (minutes)"""
    days, hms = s.split("d")
    h, m, _ = map(int, hms.split(":"))
    return int(days) * 24 * 60 + h * 60 + m


def train_minutes(frm, to, date, time_):
    """Shortest of the next 4 connections, in minutes."""
    params = {"from": frm, "to": to, "date": date, "time": time_, "limit": 4}
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return min(parse_duration(c["duration"]) for c in r.json()["connections"])


WALK = 10  # Engelberg station → Titlis valley station, minutes (each way)

t_out = train_minutes("Zürich HB", "Engelberg", "2026-12-10", "07:00")
time.sleep(1)
t_back = train_minutes("Engelberg", "Zürich HB", "2026-12-10", "16:30")

T = t_out + t_back + 2 * WALK
print(f"out {t_out} min, back {t_back} min, total {T} min = {T/60:.2f} h")

out 103 min, back 109 min, total 232 min = 3.87 h


### Compute travel cost K by pass type

In [8]:
fares = pd.read_csv("../docs/train_fares.csv")
fare = fares.loc[fares["resort"] == "Engelberg-Titlis"].iloc[0]

def train_cost(fare, travel_pass):
    """Return fare in CHF for 'none', 'Halbtax' or 'GA'."""
    if travel_pass == "GA":
        return 0.0
    if travel_pass == "Halbtax":
        return fare["fare_halbtax_return"]
    return fare["fare_full_return"]

v = 20  # example value of time, CHF per hour

for p in ["none", "Halbtax", "GA"]:
    K = train_cost(fare, p)
    print(f"{p:8}  K = {K:5.2f}   v·T = {v * T / 60:5.2f}   K + v·T = {K + v * T / 60:6.2f}")

none      K = 80.00   v·T = 77.33   K + v·T = 157.33
Halbtax   K = 40.00   v·T = 77.33   K + v·T = 117.33
GA        K =  0.00   v·T = 77.33   K + v·T =  77.33


## 8.4 Car (OpenRouteService)

In [9]:
from pathlib import Path

def read_env(path="../.env"):
    """Read KEY=value lines from .env (without printing them)."""
    env = {}
    for line in Path(path).read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip()
    return env

ORS_KEY = read_env()["ORS_KEY"]
ORS_URL = "https://api.heigit.org/openrouteservice/v2/directions/driving-car"

ZURICH_HB = [8.5403, 47.3779]                       # [lon, lat]
resort = pd.read_csv("../docs/resort_attributes.csv").iloc[0]
ENGELBERG = [resort["lon"], resort["lat"]]          # Titlis valley station

def drive(start, end):
    """One-way car route → (km, minutes)."""
    r = requests.post(ORS_URL,
                      headers={"Authorization": ORS_KEY, "Content-Type": "application/json"},
                      json={"coordinates": [start, end]},
                      timeout=30)
    r.raise_for_status()
    s = r.json()["routes"][0]["summary"]
    return s["distance"] / 1000, s["duration"] / 60

km_out, min_out = drive(ZURICH_HB, ENGELBERG)
time.sleep(2)
km_back, min_back = drive(ENGELBERG, ZURICH_HB)
print(f"out {km_out:.1f} km, {min_out:.0f} min | back {km_back:.1f} km, {min_back:.0f} min")

out 85.5 km, 76 min | back 86.4 km, 79 min
